
### Setup: foreachBatch Patterns
Creates per-exercise streaming source tables, target tables, a metrics table, and a
quarantine table for the `foreachbatch_patterns` schema. Each exercise gets its own
isolated `exN_source` / `exN_target` pair so streams cannot interfere with each other.
Checkpoints live under a Unity Catalog volume so streams are restartable.

Run automatically via `%run` from the exercise notebook.

In [0]:
CATALOG="db_code"
SCHEMA="foreachbatch_patterns"
BASE_SCHEMA="streaming"

#create volume 
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.checkpoints")

CHECKPOINT_BASE=f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"




## Per-exercise streaming source tables
Each `exN_source` is a Delta table with ~40 events. Rows include intentional edge
cases (negative `amount`, NULL `user_id`) used by the quality-routing exercises.

Schema:
| Column     | Type      | Notes |
|------------|-----------|-------|
| event_id   | STRING    | Primary key (unique within source) |
| user_id    | STRING    | Nullable; ~2 rows have NULL |
| event_type | STRING    | purchase / view / click |
| amount     | DOUBLE    | Nullable; ~2 rows are negative (bad data) |
| event_ts   | TIMESTAMP | Event timestamp |

In [0]:
# Canonical event dataset for the topic.
# 40 rows. 36 "good" rows + 4 "bad" rows for the quality-routing exercises:
#   - 2 rows with negative amount (EV-901, EV-902)
#   - 2 rows with NULL user_id (EV-903, EV-904)
events_rows = """
    ('EV-301', 'purchase', 'U-1',  50.00,  TIMESTAMP '2026-05-01 10:00:00'),
    ('EV-302', 'view',     'U-2',  0.00,   TIMESTAMP '2026-05-01 10:01:00'),
    ('EV-303', 'purchase', 'U-3',  120.50, TIMESTAMP '2026-05-01 10:02:00'),
    ('EV-304', 'click',    'U-1',  0.00,   TIMESTAMP '2026-05-01 10:03:00'),
    ('EV-305', 'purchase', 'U-2',  75.25,  TIMESTAMP '2026-05-01 10:04:00'),
    ('EV-306', 'view',     'U-4',  0.00,   TIMESTAMP '2026-05-01 10:05:00'),
    ('EV-307', 'purchase', 'U-3',  200.00, TIMESTAMP '2026-05-01 10:06:00'),
    ('EV-308', 'click',    'U-5',  0.00,   TIMESTAMP '2026-05-01 10:07:00'),
    ('EV-309', 'purchase', 'U-1',  45.00,  TIMESTAMP '2026-05-01 10:08:00'),
    ('EV-310', 'view',     'U-2',  0.00,   TIMESTAMP '2026-05-01 10:09:00'),
    ('EV-311', 'purchase', 'U-4',  310.75, TIMESTAMP '2026-05-01 10:10:00'),
    ('EV-312', 'click',    'U-3',  0.00,   TIMESTAMP '2026-05-01 10:11:00'),
    ('EV-313', 'purchase', 'U-5',  89.99,  TIMESTAMP '2026-05-01 10:12:00'),
    ('EV-314', 'view',     'U-1',  0.00,   TIMESTAMP '2026-05-01 10:13:00'),
    ('EV-315', 'purchase', 'U-2',  125.00, TIMESTAMP '2026-05-01 10:14:00'),
    ('EV-316', 'click',    'U-4',  0.00,   TIMESTAMP '2026-05-01 10:15:00'),
    ('EV-317', 'purchase', 'U-3',  60.00,  TIMESTAMP '2026-05-01 10:16:00'),
    ('EV-318', 'view',     'U-5',  0.00,   TIMESTAMP '2026-05-01 10:17:00'),
    ('EV-319', 'purchase', 'U-1',  175.50, TIMESTAMP '2026-05-01 10:18:00'),
    ('EV-320', 'click',    'U-2',  0.00,   TIMESTAMP '2026-05-01 10:19:00'),
    ('EV-321', 'purchase', 'U-4',  99.50,  TIMESTAMP '2026-05-01 10:20:00'),
    ('EV-322', 'view',     'U-3',  0.00,   TIMESTAMP '2026-05-01 10:21:00'),
    ('EV-323', 'purchase', 'U-5',  220.00, TIMESTAMP '2026-05-01 10:22:00'),
    ('EV-324', 'click',    'U-1',  0.00,   TIMESTAMP '2026-05-01 10:23:00'),
    ('EV-325', 'purchase', 'U-2',  30.00,  TIMESTAMP '2026-05-01 10:24:00'),
    ('EV-326', 'view',     'U-4',  0.00,   TIMESTAMP '2026-05-01 10:25:00'),
    ('EV-327', 'purchase', 'U-3',  410.00, TIMESTAMP '2026-05-01 10:26:00'),
    ('EV-328', 'click',    'U-5',  0.00,   TIMESTAMP '2026-05-01 10:27:00'),
    ('EV-329', 'purchase', 'U-1',  15.50,  TIMESTAMP '2026-05-01 10:28:00'),
    ('EV-330', 'view',     'U-2',  0.00,   TIMESTAMP '2026-05-01 10:29:00'),
    ('EV-331', 'purchase', 'U-4',  270.00, TIMESTAMP '2026-05-01 10:30:00'),
    ('EV-332', 'click',    'U-3',  0.00,   TIMESTAMP '2026-05-01 10:31:00'),
    ('EV-333', 'purchase', 'U-5',  55.00,  TIMESTAMP '2026-05-01 10:32:00'),
    ('EV-334', 'view',     'U-1',  0.00,   TIMESTAMP '2026-05-01 10:33:00'),
    ('EV-335', 'purchase', 'U-2',  145.75, TIMESTAMP '2026-05-01 10:34:00'),
    ('EV-336', 'purchase', 'U-3',  88.00,  TIMESTAMP '2026-05-01 10:35:00'),
    -- "Bad" rows for quality routing (filter target: amount >= 0 AND user_id IS NOT NULL)
    ('EV-901', 'purchase', 'U-1',  -10.00, TIMESTAMP '2026-05-01 11:00:00'),
    ('EV-902', 'purchase', 'U-4',  -25.50, TIMESTAMP '2026-05-01 11:01:00'),
    ('EV-903', 'purchase', NULL,   50.00,  TIMESTAMP '2026-05-01 11:02:00'),
    ('EV-904', 'view',     NULL,   0.00,   TIMESTAMP '2026-05-01 11:03:00')
"""

def create_events_source(table_name: str):
    spark.sql(f"""
        CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.{table_name} (
            event_id STRING,
            event_type STRING,
            user_id STRING,
            amount DOUBLE,
            event_ts TIMESTAMP
        ) USING DELTA
    """)
    spark.sql(f"""
        INSERT INTO {CATALOG}.{SCHEMA}.{table_name} VALUES {events_rows}
    """)

# Exercises 1-7 all stream from a copy of the canonical 40-row source.
# Each gets its own copy so simultaneous reruns of the notebook don't cause read-write
# conflicts on a shared source.
for ex in [1, 2, 3, 4, 5, 6, 7]:
    create_events_source(f"ex{ex}_source")

## Exercise 2 + 4: pre-existing target table for MERGE upsert exercises
`ex2_target` has 10 rows. Streaming events for matching `event_id`s will UPDATE
the existing rows; new `event_id`s will INSERT.

In [0]:
# Target for Exercise 2 (foreachBatch + MERGE).
# 10 rows pre-populated. Streaming source has rows for EV-301..EV-310 (overlap) plus
# EV-311..EV-336 + bad rows. MERGE on event_id should UPDATE the 10 existing rows
# (event_type/user_id/amount get the source values) and INSERT the remaining new rows.
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.ex2_target (
        event_id STRING,
        event_type STRING,
        user_id STRING,
        amount DOUBLE,
        event_ts TIMESTAMP
    ) USING DELTA
""")
spark.sql(f"""
    INSERT INTO {CATALOG}.{SCHEMA}.ex2_target VALUES
        ('EV-301', 'OLD', 'OLD', 0.01, TIMESTAMP '2020-01-01 00:00:00'),
        ('EV-302', 'OLD', 'OLD', 0.01, TIMESTAMP '2020-01-01 00:00:00'),
        ('EV-303', 'OLD', 'OLD', 0.01, TIMESTAMP '2020-01-01 00:00:00'),
        ('EV-304', 'OLD', 'OLD', 0.01, TIMESTAMP '2020-01-01 00:00:00'),
        ('EV-305', 'OLD', 'OLD', 0.01, TIMESTAMP '2020-01-01 00:00:00'),
        ('EV-306', 'OLD', 'OLD', 0.01, TIMESTAMP '2020-01-01 00:00:00'),
        ('EV-307', 'OLD', 'OLD', 0.01, TIMESTAMP '2020-01-01 00:00:00'),
        ('EV-308', 'OLD', 'OLD', 0.01, TIMESTAMP '2020-01-01 00:00:00'),
        ('EV-309', 'OLD', 'OLD', 0.01, TIMESTAMP '2020-01-01 00:00:00'),
        ('EV-310', 'OLD', 'OLD', 0.01, TIMESTAMP '2020-01-01 00:00:00')
""")

## Exercise 3: two target tables (raw + clean) for multi-sink writes


In [0]:
# Raw target: every event in the batch, no transformation
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.ex3_target_raw (
        event_id STRING,
        event_type STRING,
        user_id STRING,
        amount DOUBLE,
        event_ts TIMESTAMP
    ) USING DELTA
""")

# Clean target: only event_type='purchase' AND amount > 0
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.ex3_target_clean (
        event_id STRING,
        event_type STRING,
        user_id STRING,
        amount DOUBLE,
        event_ts TIMESTAMP
    ) USING DELTA
""")

## Exercise 4: idempotent target table (tracks batch_id)
Empty target. User's foreachBatch tags each row with `batch_id` and uses a MERGE
that skips already-applied batches, so calling the function twice with the same
`batch_id` is a no-op.

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.ex4_target (
        event_id STRING,
        event_type STRING,
        user_id STRING,
        amount DOUBLE,
        event_ts TIMESTAMP,
        batch_id LONG
    ) USING DELTA
""")

## Exercise 5: quarantine + main target for quality routing

In [0]:
# Main target receives the "good" rows (amount >= 0 AND user_id IS NOT NULL)
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.ex5_target (
        event_id STRING,
        event_type STRING,
        user_id STRING,
        amount DOUBLE,
        event_ts TIMESTAMP
    ) USING DELTA
""")

# Quarantine receives the "bad" rows with a `reason` column explaining why
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.ex5_quarantine (
        event_id STRING,
        event_type STRING,
        user_id STRING,
        amount DOUBLE,
        event_ts TIMESTAMP,
        reason STRING
    ) USING DELTA
""")

## Exercise 6: metrics table for running-metrics pattern

In [0]:
# One row per microbatch with operational metrics.
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.ex6_metrics (
        batch_id LONG,
        rows_in LONG,
        rows_good LONG,
        rows_quarantined LONG,
        processed_at TIMESTAMP
    ) USING DELTA
""")

# Main target for Exercise 6 (clean rows only, like Exercise 5)
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.ex6_target (
        event_id STRING,
        event_type STRING,
        user_id STRING,
        amount DOUBLE,
        event_ts TIMESTAMP
    ) USING DELTA
""")

## Exercise 7: materialized batch_df target + quarantine
The exercise reads the batch DataFrame twice (once for a MERGE upsert into the
main target, once for routing rejects into quarantine). Without materializing the
batch via collect(), Spark may recompute the batch DataFrame.

In [0]:
# Pre-populated main target (will be upserted via MERGE)
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.ex7_target (
        event_id STRING,
        event_type STRING,
        user_id STRING,
        amount DOUBLE,
        event_ts TIMESTAMP
    ) USING DELTA
""")
spark.sql(f"""
    INSERT INTO {CATALOG}.{SCHEMA}.ex7_target VALUES
        ('EV-301', 'OLD', 'OLD', 0.01, TIMESTAMP '2020-01-01 00:00:00'),
        ('EV-302', 'OLD', 'OLD', 0.01, TIMESTAMP '2020-01-01 00:00:00'),
        ('EV-303', 'OLD', 'OLD', 0.01, TIMESTAMP '2020-01-01 00:00:00')
""")

# Quarantine for the rejects
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.ex7_quarantine (
        event_id STRING,
        event_type STRING,
        user_id STRING,
        amount DOUBLE,
        event_ts TIMESTAMP
    ) USING DELTA
""")